# 02 — Micro-Doppler Visualization

Visualise the signal processing pipeline:
1. Raw time-domain signal
2. Conventional spectrogram
3. Proposed regularized complex-log-Fourier transform
4. Side-by-side comparison of feature representations

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt

from radar_drone_vision.datasets.synthetic import SyntheticGenerator
from radar_drone_vision.signal.framing import frame_signal
from radar_drone_vision.signal.spectrogram import compute_spectrogram
from radar_drone_vision.signal.complex_log_fft import regularized_complex_log_fft, FEATURE_MODES

## 1. Generate a sample UAV and bird signal

In [ ]:
gen = SyntheticGenerator(seed=7, sample_duration_s=0.5, sample_rate_hz=2000.0)
uav_sample = gen.generate_uav_samples(n=1)[0]
bird_sample = gen.generate_bird_samples(n=1)[0]

print(f"UAV signal shape:  {uav_sample.signal.shape}")
print(f"Bird signal shape: {bird_sample.signal.shape}")

## 2. Frame the signal and compute spectrogram

In [ ]:
FRAME_SIZE = 128
HOP_SIZE = 64
N_FFT = 128

for name, sample in [("UAV", uav_sample), ("Bird", bird_sample)]:
    frames = frame_signal(sample.signal, frame_size=FRAME_SIZE, hop_size=HOP_SIZE)
    spec = compute_spectrogram(frames, n_fft=N_FFT)
    print(f"{name}: frames={frames.shape}, spectrogram={spec.shape}")

## 3. Compute proposed feature (all modes)

In [ ]:
frames_uav = frame_signal(uav_sample.signal, frame_size=FRAME_SIZE, hop_size=HOP_SIZE)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, mode in zip(axes.ravel(), FEATURE_MODES):
    feat = regularized_complex_log_fft(frames_uav, n_fft=N_FFT, feature_mode=mode)
    im = ax.imshow(feat, aspect="auto", origin="lower", cmap="viridis")
    ax.set_title(f"{mode}  shape={feat.shape}")
    ax.set_xlabel("Feature bin")
    ax.set_ylabel("Frame")
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle("UAV — Proposed Feature (all modes)", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Side-by-side: spectrogram vs. proposed feature

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for col, (name, sample) in enumerate([("UAV", uav_sample), ("Bird", bird_sample)]):
    frames = frame_signal(sample.signal, frame_size=FRAME_SIZE, hop_size=HOP_SIZE)
    spec = compute_spectrogram(frames, n_fft=N_FFT)
    proposed = regularized_complex_log_fft(frames, n_fft=N_FFT, feature_mode="magnitude_only")

    axes[0, col].imshow(spec, aspect="auto", origin="lower", cmap="inferno")
    axes[0, col].set_title(f"{name} — Spectrogram")

    axes[1, col].imshow(proposed, aspect="auto", origin="lower", cmap="viridis")
    axes[1, col].set_title(f"{name} — Proposed (magnitude_only)")

for ax in axes.ravel():
    ax.set_xlabel("Frequency bin")
    ax.set_ylabel("Frame")

plt.tight_layout()
plt.show()